In [1]:
# ============================================================
# 🔧 SYSTEM & UTILITIES
# ============================================================
import os
import gc
import re
import io
import psutil
import json
import joblib
import pickle
import random
import requests
import itertools
import warnings
import holidays
import urllib.request as urlreq
from tqdm import tqdm
from dotenv import load_dotenv
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta

# Suppress warnings
warnings.filterwarnings('ignore')

# ============================================================
# 📦 CORE PYTHON & DATA HANDLING
# ============================================================
import numpy as np
import pandas as pd
import datetime as dt

# ============================================================
# 📊 EXPLORATORY DATA ANALYSIS (EDA) & VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import missingno as msno
from pandas.plotting import lag_plot, autocorrelation_plot
from scipy import stats
from math import sqrt
from IPython.display import display, HTML

# Enable inline plotting
%matplotlib inline

# ============================================================
# ⚙️ DATA PREPROCESSING & FEATURE ENGINEERING
# ============================================================
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# ============================================================
# 🧠 MACHINE LEARNING MODELS
# ============================================================
from sklearn.model_selection import (
    train_test_split, TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score, make_scorer
)
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# ============================================================
# 🔮 TIME SERIES & FORECASTING
# ============================================================
from statsmodels.tsa.api import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ============================================================
# 🧬 DEEP LEARNING (TENSORFLOW / KERAS)
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Bidirectional, BatchNormalization,
    Conv1D, MaxPooling1D, Flatten
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from tensorflow.keras.utils import plot_model
from tensorflow.keras import backend as K
import keras_tuner as kt

# ============================================================
# 🧮 STATISTICAL ANALYSIS
# ============================================================
import scipy.stats as stats
from scipy.stats import zscore, pearsonr, spearmanr

# ============================================================
# 🗂 DASHBOARDING & FRONT-END (OPTIONAL)
# ============================================================
import streamlit as st

# ============================================================
# ✅ SETUP COMPLETE
# ============================================================
print("✅ All essential libraries imported successfully.")


✅ All essential libraries imported successfully.


In [22]:
df = pd.read_csv(r'C:\Users\TPWODL\New folder_Content\DeepLearning_TimeSeries_LSTM_End_To_End\data\raw\Energy Demand Hourly.csv')

In [23]:
df.head(2)

,date,megawatthours
0,01-07-2015 05:00,162827
1,01-07-2015 06:00,335153


In [24]:
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y %H:%M', errors='coerce')

In [25]:
# ===== CORE TIME FEATURES =====
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_month'] = df['date'].dt.day
df['day_of_year'] = df['date'].dt.dayofyear

In [27]:
df

,date,megawatthours,hour,day_of_week,day_of_month,day_of_year
0,2015-07-01 05:00:00,162827,5,2,1,182
1,2015-07-01 06:00:00,335153,6,2,1,182
2,2015-07-01 07:00:00,333837,7,2,1,182
3,2015-07-01 08:00:00,398386,8,2,1,182
4,2015-07-01 09:00:00,388954,9,2,1,182
...,...,...,...,...,...,...
58931,2022-03-21 16:00:00,433344,16,0,21,80
58932,2022-03-21 17:00:00,429156,17,0,21,80
58933,2022-03-21 18:00:00,426496,18,0,21,80
58934,2022-03-21 19:00:00,423393,19,0,21,80


In [29]:
import pandas as pd
import numpy as np
import json
from statsmodels.tsa.seasonal import seasonal_decompose
from datetime import datetime

def analyze_seasonal_decomposition(df, column_name, model='additive', period=12):
    """
    Perform seasonal decomposition on time series data and return JSON report.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with datetime index
    column_name : str
        Name of the column to analyze
    model : str, default='additive'
        Type of seasonal component ('additive' or 'multiplicative')
    period : int, default=12
        Number of observations per cycle
        
    Returns:
    --------
    dict : JSON-formatted report with decomposition results
    """
    
    # Validate inputs
    if column_name not in df.columns:
        return {"error": f"Column '{column_name}' not found in DataFrame"}
    
    if not isinstance(df.index, pd.DatetimeIndex):
        return {"error": "DataFrame must have a DatetimeIndex"}
    
    # Extract time series
    ts = df[column_name].dropna()
    
    if len(ts) < 2 * period:
        return {"error": f"Insufficient data points. Need at least {2*period}, got {len(ts)}"}
    
    # Perform decomposition
    try:
        result = seasonal_decompose(ts, model=model, period=period)
    except Exception as e:
        return {"error": f"Decomposition failed: {str(e)}"}
    
    # Extract components
    trend = result.trend.dropna()
    seasonal = result.seasonal.dropna()
    residual = result.resid.dropna()
    
    # Calculate statistics
    report = {
        "metadata": {
            "analysis_date": datetime.now().isoformat(),
            "column_analyzed": column_name,
            "model_type": model,
            "period": period,
            "total_observations": len(ts),
            "date_range": {
                "start": ts.index.min().isoformat(),
                "end": ts.index.max().isoformat()
            }
        },
        "original_series": {
            "mean": float(ts.mean()),
            "std": float(ts.std()),
            "min": float(ts.min()),
            "max": float(ts.max()),
            "median": float(ts.median())
        },
        "trend_component": {
            "mean": float(trend.mean()),
            "std": float(trend.std()),
            "min": float(trend.min()),
            "max": float(trend.max()),
            "non_null_count": int(len(trend)),
            "trend_direction": "increasing" if trend.iloc[-1] > trend.iloc[0] else "decreasing",
            "change_percent": float(((trend.iloc[-1] - trend.iloc[0]) / trend.iloc[0]) * 100) if trend.iloc[0] != 0 else None
        },
        "seasonal_component": {
            "mean": float(seasonal.mean()),
            "std": float(seasonal.std()),
            "min": float(seasonal.min()),
            "max": float(seasonal.max()),
            "amplitude": float(seasonal.max() - seasonal.min()),
            "non_null_count": int(len(seasonal))
        },
        "residual_component": {
            "mean": float(residual.mean()),
            "std": float(residual.std()),
            "min": float(residual.min()),
            "max": float(residual.max()),
            "non_null_count": int(len(residual)),
            "outliers_count": int(np.sum(np.abs(residual) > 3 * residual.std()))
        },
        "decomposition_quality": {
            "residual_variance_ratio": float((residual.var() / ts.var()) * 100),
            "seasonal_strength": float(1 - (residual.var() / (seasonal + residual).var())) if (seasonal + residual).var() != 0 else None,
            "trend_strength": float(1 - (residual.var() / (trend + residual).var())) if (trend + residual).dropna().var() != 0 else None
        },
        "sample_data": {
            "trend_head": trend.head(5).to_dict(),
            "seasonal_head": seasonal.head(5).to_dict(),
            "residual_head": residual.head(5).to_dict()
        }
    }
    
    return report


# Example usage:
# Assuming you have a DataFrame 'df' with datetime index and 'megawatthours' column
# report = analyze_seasonal_decomposition(df, 'megawatthours', model='additive', period=12)
# 
# # Print formatted JSON
# print(json.dumps(report, indent=2))
# 
# # Save to file
# with open('decomposition_report.json', 'w') as f:
#     json.dump(report, f, indent=2)

In [30]:
report = analyze_seasonal_decomposition(df, 'megawatthours', model='additive', period=12)


In [31]:
report

{'error': 'DataFrame must have a DatetimeIndex'}

In [32]:
import pandas as pd
import numpy as np
import json
from statsmodels.tsa.seasonal import seasonal_decompose
from datetime import datetime

def analyze_seasonal_decomposition(df, column_name, date_column=None, model='additive', period=12):
    """
    Perform seasonal decomposition on time series data and return JSON report.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with time series data
    column_name : str
        Name of the column to analyze
    date_column : str, optional
        Name of the date column to use as index (if not already indexed)
    model : str, default='additive'
        Type of seasonal component ('additive' or 'multiplicative')
    period : int, default=12
        Number of observations per cycle
        
    Returns:
    --------
    dict : JSON-formatted report with decomposition results
    """
    
    # Make a copy to avoid modifying original
    df_copy = df.copy()
    
    # Validate inputs
    if column_name not in df_copy.columns:
        return {"error": f"Column '{column_name}' not found in DataFrame"}
    
    # Convert index to datetime if needed
    if not isinstance(df_copy.index, pd.DatetimeIndex):
        if date_column is not None:
            if date_column not in df_copy.columns:
                return {"error": f"Date column '{date_column}' not found in DataFrame"}
            try:
                df_copy[date_column] = pd.to_datetime(df_copy[date_column])
                df_copy = df_copy.set_index(date_column)
                df_copy = df_copy.sort_index()
            except Exception as e:
                return {"error": f"Failed to convert '{date_column}' to datetime: {str(e)}"}
        else:
            # Try to find a date column automatically
            date_cols = [col for col in df_copy.columns if 'date' in col.lower() or 'time' in col.lower()]
            if date_cols:
                try:
                    df_copy[date_cols[0]] = pd.to_datetime(df_copy[date_cols[0]])
                    df_copy = df_copy.set_index(date_cols[0])
                    df_copy = df_copy.sort_index()
                except Exception as e:
                    return {"error": f"Auto-detected date column '{date_cols[0]}' conversion failed. Please specify 'date_column' parameter: {str(e)}"}
            else:
                return {"error": "No DatetimeIndex found. Please specify 'date_column' parameter or ensure DataFrame has a datetime index"}
    
    # Extract time series
    ts = df_copy[column_name].dropna()
    
    if len(ts) < 2 * period:
        return {"error": f"Insufficient data points. Need at least {2*period}, got {len(ts)}"}
    
    # Perform decomposition
    try:
        result = seasonal_decompose(ts, model=model, period=period)
    except Exception as e:
        return {"error": f"Decomposition failed: {str(e)}"}
    
    # Extract components
    trend = result.trend.dropna()
    seasonal = result.seasonal.dropna()
    residual = result.resid.dropna()
    
    # Calculate statistics
    report = {
        "metadata": {
            "analysis_date": datetime.now().isoformat(),
            "column_analyzed": column_name,
            "model_type": model,
            "period": period,
            "total_observations": len(ts),
            "date_range": {
                "start": ts.index.min().isoformat(),
                "end": ts.index.max().isoformat()
            }
        },
        "original_series": {
            "mean": float(ts.mean()),
            "std": float(ts.std()),
            "min": float(ts.min()),
            "max": float(ts.max()),
            "median": float(ts.median())
        },
        "trend_component": {
            "mean": float(trend.mean()),
            "std": float(trend.std()),
            "min": float(trend.min()),
            "max": float(trend.max()),
            "non_null_count": int(len(trend)),
            "trend_direction": "increasing" if trend.iloc[-1] > trend.iloc[0] else "decreasing",
            "change_percent": float(((trend.iloc[-1] - trend.iloc[0]) / trend.iloc[0]) * 100) if trend.iloc[0] != 0 else None
        },
        "seasonal_component": {
            "mean": float(seasonal.mean()),
            "std": float(seasonal.std()),
            "min": float(seasonal.min()),
            "max": float(seasonal.max()),
            "amplitude": float(seasonal.max() - seasonal.min()),
            "non_null_count": int(len(seasonal))
        },
        "residual_component": {
            "mean": float(residual.mean()),
            "std": float(residual.std()),
            "min": float(residual.min()),
            "max": float(residual.max()),
            "non_null_count": int(len(residual)),
            "outliers_count": int(np.sum(np.abs(residual) > 3 * residual.std()))
        },
        "decomposition_quality": {
            "residual_variance_ratio": float((residual.var() / ts.var()) * 100),
            "seasonal_strength": float(1 - (residual.var() / (seasonal + residual).var())) if (seasonal + residual).var() != 0 else None,
            "trend_strength": float(1 - (residual.var() / (trend + residual).var())) if (trend + residual).dropna().var() != 0 else None
        },
        "sample_data": {
            "trend_head": trend.head(5).to_dict(),
            "seasonal_head": seasonal.head(5).to_dict(),
            "residual_head": residual.head(5).to_dict()
        }
    }
    
    return report


# Example usage:
# 
# Option 1: If you have a date column (e.g., 'date')
# report = analyze_seasonal_decomposition(df, 'megawatthours', date_column='date', model='additive', period=12)
# 
# Option 2: If your DataFrame already has a datetime index
# report = analyze_seasonal_decomposition(df, 'megawatthours', model='additive', period=12)
# 
# Option 3: Let the function auto-detect the date column
# report = analyze_seasonal_decomposition(df, 'megawatthours', model='additive', period=12)
#
# # Print formatted JSON
# print(json.dumps(report, indent=2))
# 
# # Save to file
# with open('decomposition_report.json', 'w') as f:
#     json.dump(report, f, indent=2)

In [33]:
report = analyze_seasonal_decomposition(df, 'megawatthours', date_column='date', model='additive', period=12)


In [ ]:
def save_json(data: Dict[str, Any], file_path: str) -> None:
    """
    Save dictionary data to JSON file.
    
    Args:
        data: Dictionary to save
        file_path: Destination file path
        
    Raises:
        DataIngestionException: If saving fails
    """
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=4)
        
        logger.info(f"JSON data saved to: {file_path}")
        
    except Exception as e:
        raise ProjectException(e, sys)

In [34]:
report

{'metadata': {'analysis_date': '2025-11-23T04:20:47.465725',
  'column_analyzed': 'megawatthours',
  'model_type': 'additive',
  'period': 12,
  'total_observations': 58936,
  'date_range': {'start': '2015-07-01T05:00:00',
   'end': '2022-03-21T20:00:00'}},
 'original_series': {'mean': 455225.233032442,
  'std': 74804.55001857976,
  'min': 162827.0,
  'max': 719649.0,
  'median': 443225.0},
 'trend_component': {'mean': 455238.8149622112,
  'std': 61953.69241830488,
  'min': 321901.0,
  'max': 679911.8750000001,
  'non_null_count': 58924,
  'trend_direction': 'increasing',
  'change_percent': 0.6272683569072744},
 'seasonal_component': {'mean': -0.5148669556278943,
  'std': 11608.880436242383,
  'min': -15073.484279557346,
  'max': 17560.910072459472,
  'amplitude': 32634.394352016818,
  'non_null_count': 58936},
 'residual_component': {'mean': 0.8215249748830205,
  'std': 20383.994930964152,
  'min': -77389.47979036796,
  'max': 124841.80181180447,
  'non_null_count': 58924,
  'outlier